# Advisor reproducibility — start here

Select the installed **pytorch** kernel, edit the configuration below, then **Run All**.
This one notebook prepares the local inputs, generates/reconstructs complex H and A,
and renders all five essential figures **inline**. No prior plot files or CLI runs needed.

- **saved_replay** (default): reconstruct the frozen physical H from saved paths,
  recompute each complex A, and replot completed BER evidence. **No BER campaign is rerun.**
- **fresh**: generate new DD channels and run a bounded paired BER check, clearly
  labeled as a small simulation rather than a reproduction of the completed campaign.

One fixed N64 matrix labeled **Learned**, with OTFS and AFDM throughout.
The reference was retained after a post-hoc tradeoff review; no universal winner claim.
Internal studies stay separate. Training is a later workflow and is not implemented here.


In [ ]:
# CONFIGURATION — edit this cell, then Run All from the top.
from pathlib import Path
import os, sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "waveforms").is_dir() and (p / "pyproject.toml").is_file())
sys.path.insert(0, str(ROOT))
for key in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[key] = "1"
os.environ["MPLCONFIGDIR"] = str(ROOT / "local/.matplotlib")

MODE = "saved_replay"                 # "saved_replay" or "fresh"
ARTIFACT_DIR = ROOT / "local/artifacts"
ARTIFACT_BUNDLE = ROOT / "local/advisor-n64-artifacts.tar.gz"
SAVE_RUN = True                        # Optional local archive; figures are inline either way.

MODEL = dict(n=64, paths=3, delay_range=(0.0, 4.0), doppler_range=(-0.75, 0.75))
DISPLAY = dict(basis_columns=(0, 11, 15, 42, 44, 52), snr_max_db=25.0)  # None = full SNR range.

FRESH = None                           # Must stay None in saved_replay.
if MODE == "fresh":
    FRESH = dict(seed=20261116, matrix_channels=2, matrix_index=0,
                 ber_channels=8, blocks_per_channel=2, snrs_db=(0.0, 10.0, 18.0, 25.0))


## 1. Validate settings and prepare inputs

The selected Learned export fixes **N=64**; no resizing or alternative checkpoint selection.
Replay rejects model changes because the saved BER curves belong to nominal P3,
delay[0,4], Doppler[-.75,.75]. Display columns/SNR limit may change without simulating.
Fresh mode permits P1–6, delay limits within[0,8], Doppler limits within[-2,2],
1–16 matrix channels,2–64 BER channels,1–16 blocks and2–8 SNR points;
BER channels × blocks × SNR points must be ≤4096. Unsupported keys/settings fail clearly.

If ARTIFACT_DIR is absent, the supplied local bundle is provisioned and checksum-verified
here automatically. No downloads or original-project access. CPU float64/complex128,
one thread, was explicitly authorized for this package; no device/precision fallback.


In [ ]:
from IPython.display import display, JSON
import pandas as pd
from waveforms.notebook_workflow import start_workflow

workflow = start_workflow(mode=MODE, model=MODEL, display=DISPLAY, fresh=FRESH,
                          artifacts=ARTIFACT_DIR, bundle=ARTIFACT_BUNDLE,
                          workspace=ROOT, save_run=SAVE_RUN)
display(JSON(workflow.summary(), expanded=True))


## 2. Physical channel and complex effective matrices

DD convention: negative delay/Doppler signs, delay first;
Hraw=ΣhₚΔ(νₚ)Π(τₚ), H=Hraw/√(‖Hraw‖F²/64), A=WᴴHW.
Replay reconstructs the existing illustrative channel from its paths; fresh mode
uses the configured seed/model and displayed channel index. This matrix bank and
the fresh BER bank have separate RNG namespaces. H, W and A remain available as
complex tensors in memory for inspection; the heatmap shows entry power only.


In [ ]:
channels, A = workflow.generate_channels()
H, Hraw, W = channels.h, channels.raw, workflow.ws
display(workflow.matrix_summary())
i = workflow.matrix_index
display(pd.DataFrame({"delay": channels.tau[i].numpy(), "Doppler": channels.nu[i].numpy(),
                      "complex gain": channels.gains[i].numpy()}))
print("Channel normalization:", float(channels.scale[i]))
workflow.show("channel_matrices_native")


## 3. Time-domain and frequency-domain basis figures

The same Learned matrix is used everywhere. DISPLAY.basis_columns changes which
columns are shown, not the waveform itself. All three bases use the same indices;
DFT powers are absolute, without per-column peak normalization.


In [ ]:
workflow.show("basis_time")
workflow.show("basis_frequency")


## 4. BER versus SNR — read the mode label

**Replay loads saved counts/intervals; it does not simulate BER.** Fresh mode
computes new paired H/bits/time-domain-noise observations for every method, with
the configured budgets. Fresh curves are smoke results with descriptive channel-SE
bands; they are not the saved campaign's bootstrap intervals or precise performance claims.

Raw QPSK/Gray16QAM slicing, Es/N0. Fixed and full receivers stay separate;
actual observations always use full A. Fixed Q24 positions remain the original
nominal discovery masks, even when the fresh channel model changes: no hidden
support recalibration or training. Zero errors are unresolved; the table retains
counts, resolution and conservative independent-channel upper bounds.


In [ ]:
ber = workflow.evaluate_ber()
print(workflow.provenance["BER_status"])
display(ber[["modulation", "snr_db", "waveform", "receiver", "channels", "bits", "errors", "ber",
             "ci_low", "ci_high", "zero_cluster_upper95", "detection_limit"]])
workflow.show("ber_QPSK_raw")
workflow.show("ber_16QAM_raw")


## Complete

All five essential figures are visible above. The optional archive preserves complex
matrices, paths, masks, seeds, counts, settings and source hashes under local/;
you do not need to open it to review the results. Re-running creates a new archive
rather than overwriting evidence. After changing configuration, Run All again.
No training or new research workflow starts here.


In [ ]:
display(JSON(workflow.finish(), expanded=True))
